In [1]:
import sys
from pathlib import Path
repo_root = Path().resolve().parent  
sys.path.append(str(repo_root))

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score, mean_absolute_percentage_error
from sklearn.cluster import KMeans
import joblib
import plotly.graph_objects as go
import plotly.express as px 
from sklearn.pipeline import Pipeline 
import matplotlib.pyplot as plt
import seaborn as sns
from src.data.Tenth_Best_Time_Estimator import TenthBestTimeEstimator


In [3]:
raw_parquet = pd.read_parquet('../data/raw/reunion_segments.parquet')
df_parquet = pd.DataFrame(raw_parquet)
df_parquet = df_parquet.rename(columns={"id": "segment_id"})
df_parquet = df_parquet.drop_duplicates(subset=['segment_id'])
df_parquet = df_parquet.dropna(subset=['activity_type'])
df_parquet = df_parquet.dropna(subset=['altitude_profile'])

In [4]:
model_path = repo_root / "src" / "models" / "missing_tenth_best_time_estimator.pkl"
tte = TenthBestTimeEstimator(model_path)

df_parquet_missing_tbt = df_parquet[df_parquet['tenth_best_time'].isna()]
df_parquet_adding_tbt = tte.add_tenth_best_time(df_parquet_missing_tbt)
df_existing = df_parquet.dropna(subset=['tenth_best_time'])

df_parquet_clean = pd.concat([df_existing, df_parquet_adding_tbt], ignore_index=True)
df_parquet_clean = df_parquet_clean.drop(columns=["log_efforts", "log_athletes"])

In [5]:
df_parquet_ride = df_parquet_clean[df_parquet_clean['activity_type'] == 'Ride']

In [6]:
df_parquet_ride = df_parquet_ride.copy()
df_parquet_ride.loc[:, "best_time_speed"] = df_parquet_ride["distance"] / df_parquet_ride["best_time"]
df_parquet_ride.loc[:, "average_top_10_speed"] = df_parquet_ride["distance"] / df_parquet_ride["average_top_10_time"]
df_parquet_ride.loc[:, "tenth_best_time_speed"] = df_parquet_ride["distance"] / df_parquet_ride["tenth_best_time"]

In [7]:
raw_csv = pd.read_csv('../data/raw/reunion_segments.csv')

In [8]:
raw_manually_labeled = pd.read_csv('../data/processed/segments_manually_labeled.csv')
df_manually_labeled = pd.DataFrame(raw_manually_labeled)
df_manually_labeled_t1 = df_manually_labeled[df_manually_labeled['technicality'] == 1]

raw_road_naming_rides = pd.read_csv('../data/processed/road_naming_ride_T1.csv')
df_road_naming_rides = pd.DataFrame(raw_road_naming_rides)

In [16]:
df_manually_labeled_combined = pd.concat([df_manually_labeled_t1, df_road_naming_rides], ignore_index=True)
df_manually_labeled_combined = df_manually_labeled_combined.drop_duplicates(subset='segment_id', keep='first')
df = df_parquet_ride.merge(df_manually_labeled_combined, on='segment_id', how='inner')

### Modèle basique

In [17]:
X = df.drop(columns=['best_time', 'name', 'average_top_10_time', 'tenth_best_time', 'activity_type', 'segment_id', 'technicality', 'altitude_profile', 'distance_profile', 'coordinates'])
y = df['best_time']

In [18]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training samples: {X_train.shape[0]}, Testing samples: {X_test.shape[0]}")

Training samples: 127, Testing samples: 32


In [19]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])
pipeline.fit(X_train, y_train)

,steps,"[('scaler', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None


In [20]:
y_pred = pipeline.predict(X_test)
print(f'Mean Absolute Error: {mean_absolute_error(y_test, y_pred):.2f} seconds')
print(f'Mean Absolute Percentage Error: {mean_absolute_percentage_error(y_test, y_pred)*100:.2f} %')
print(f'R^2 Score: {r2_score(y_test, y_pred):.2f}')

Mean Absolute Error: 128.04 seconds
Mean Absolute Percentage Error: 125.32 %
R^2 Score: 0.96


In [21]:
# plot actual vs predicted
fig = go.Figure()
fig.add_trace(go.Scatter(x=y_test, y=y_pred, mode='markers', name='Pred vs Act', marker=dict(color='blue', size=10, opacity=0.7, symbol='cross')))
fig.add_trace(go.Scatter(x=[y_test.min(), y_test.max()], y=[y_test.min(), y_test.max()], mode='lines', name='Ideal', line=dict(color='red', dash='dash')))
fig.update_layout(title='Actual vs Predicted Ride Times',   
                  xaxis_title='Actual Best Time (seconds)', 
                  yaxis_title='Predicted Best Time (seconds)')
fig.show()